In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df.shape , df_test.shape

((2000, 3074), (2000, 3073))

In [ ]:
X = df.drop(columns = ['id','target'], axis=1)
y = df['target']
X_test = df_test.drop('id',axis=1).copy()

X.shape , X_test.shape , y.shape

((2000, 3072), (2000, 3072), (2000,))

In [ ]:
X_train , X_val , y_train , y_val = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)

X_train.shape , X_val.shape , y_train.shape , y_val.shape

((1600, 3072), (400, 3072), (1600,), (400,))

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=1000)
X_train = selector.fit_transform(X_train, y_train)
X_val = selector.transform(X_val)
X_test = selector.transform(X_test)

In [ ]:
X_train, X_val, X_test = pd.DataFrame(X_train), pd.DataFrame(X_val), pd.DataFrame(X_test)

In [ ]:
X_train.shape , X_val.shape , X_test.shape

((1600, 1000), (400, 1000), (2000, 1000))

In [ ]:
model = RandomForestClassifier(n_estimators=1000, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=1000, random_state=42)

In [ ]:
y_proba = model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

thresholds = np.arange(0.0, 1.01, 0.01)

best_threshold = 0
best_bal_acc = 0

for t in thresholds:
    y_pred = (y_proba > t).astype(int)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best balanced accuracy:", best_bal_acc)

Best threshold: 0.1
Best balanced accuracy: 0.6763888888888889


In [ ]:
y_pred = (y_proba > best_threshold).astype(int)

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
print(classification_report(y_val, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_val, y_pred))
print("F1 score:", f1_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("ROC AUC score:", roc_auc_score(y_val, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.68      0.79       360
           1       0.19      0.68      0.30        40

    accuracy                           0.68       400
   macro avg       0.57      0.68      0.54       400
weighted avg       0.87      0.68      0.74       400

Balanced accuracy: 0.6763888888888889
F1 score: 0.29508196721311475
Precision: 0.1888111888111888
Recall: 0.675
ROC AUC score: 0.6763888888888889
Confusion matrix:
 [[244 116]
 [ 13  27]]


In [ ]:
#Hyper parameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

param_dist = {
    'n_estimators': [100, 200, 1000],
    'max_depth': [10, 20, 30],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}
scorer = make_scorer(balanced_accuracy_score)
search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,
    scoring=scorer,
    cv=3,
    n_jobs=-1,
    verbose=1
)


search.fit(X_train, y_train)

print("Best Params:", search.best_params_)
print("Best Balanced Accuracy:", search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Params: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 30, 'class_weight': 'balanced'}
Best Balanced Accuracy: 0.5059420859538785


In [ ]:
best_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',)
best_model.fit(X_train, y_train)

y_proba = best_model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

thresholds = np.arange(0.0, 1.01, 0.01)

best_threshold = 0
best_bal_acc = 0

for t in thresholds:
    y_pred = (y_proba > t).astype(int)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best balanced accuracy:", best_bal_acc)

Best threshold: 0.07
Best balanced accuracy: 0.6944444444444444


In [ ]:
y_pred = (y_proba > best_threshold).astype(int)

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
print(classification_report(y_val, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_val, y_pred))
print("F1 score:", f1_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("ROC AUC score:", roc_auc_score(y_val, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_pred))
print('Best Params :', )

              precision    recall  f1-score   support

           0       0.99      0.44      0.61       360
           1       0.16      0.95      0.27        40

    accuracy                           0.49       400
   macro avg       0.57      0.69      0.44       400
weighted avg       0.90      0.49      0.57       400

Balanced accuracy: 0.6944444444444444
F1 score: 0.2714285714285714
Precision: 0.15833333333333333
Recall: 0.95
ROC AUC score: 0.6944444444444444
Confusion matrix:
 [[158 202]
 [  2  38]]
Best Params :


In [ ]:
y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba > best_threshold).astype(int)


In [ ]:
pd.DataFrame({'id': df_test['id'], 'target': y_test_pred}).to_csv('submission_rf.csv', index=False)